In [ ]:
%cd /kaggle/working
!git clone https://github.com/yyouretoast/deepfake-detection.git || (cd deepfake-detection && git fetch origin && git reset --hard origin/main)
%cd /kaggle/working/deepfake-detection
!pip install -q -r requirements.txt

In [ ]:
!accelerate launch --multi_gpu --mixed_precision fp16 --num_processes 2 \
    scripts/train_dual_stream_ddp.py \
    --epochs 8 \
    --batch_size 16 \
    --frequency_backbone resse \
    --hardened \
    --save_path /kaggle/working/dual_stream_best.pth

In [ ]:
!python scripts/evaluate_test_set.py \
    --weights_path /kaggle/working/dual_stream_best.pth \
    --save_calibrated /kaggle/working/dual_stream_calibrated.pth

In [ ]:
# Train Upgraded Dual-Path Bi-GRU Temporal Consistency Head (~12 min)
%cd /kaggle/working/deepfake-detection
!git fetch origin && git reset --hard origin/main
!python scripts/train_temporal_head.py \
    --backbone_weights /kaggle/working/dual_stream_calibrated.pth \
    --save_path /kaggle/working/temporal_head_best.pth \
    --epochs 5 \
    --batch_size 8 \
    --seq_len 8 \
    --stride 2 \
    --patience 2


In [ ]:
# Evaluate Temporal Test Set with Calibrated Optimal Thresholding
%cd /kaggle/working/deepfake-detection
!python scripts/evaluate_temporal_test_set.py \
    --backbone_weights /kaggle/working/dual_stream_calibrated.pth \
    --temporal_weights /kaggle/working/temporal_head_best.pth \
    --output_json /kaggle/working/temporal_test_predictions.json


In [ ]:
!python scripts/export_test_predictions.py \
    --checkpoint /kaggle/working/dual_stream_calibrated.pth \
    --output_json /kaggle/working/test_predictions.json

In [ ]:
!python scripts/evaluate_subdomain_breakdown.py \
    --weights_path /kaggle/working/dual_stream_calibrated.pth \
    --output_json /kaggle/working/subdomain_results.json


In [ ]:
!python scripts/evaluate_robustness.py \
    --checkpoint /kaggle/working/dual_stream_calibrated.pth \
    --output_json /kaggle/working/robustness_results.json

In [ ]:
# LOTO Fold 2: Face2Face (~48 min)
%cd /kaggle/working/deepfake-detection
!git fetch origin && git reset --hard origin/main
!accelerate launch --multi_gpu --mixed_precision fp16 --num_processes 2 --main_process_port 29501 \
    scripts/train_loto_experiment.py \
    --holdout face2face \
    --epochs 3 \
    --batch_size 16 \
    --num_workers 0 \
    --frequency_backbone resse \
    --hardened


In [ ]:
# LOTO Fold 3: FaceSwap (~48 min)
%cd /kaggle/working/deepfake-detection
!git fetch origin && git reset --hard origin/main
!accelerate launch --multi_gpu --mixed_precision fp16 --num_processes 2 --main_process_port 29502 \
    scripts/train_loto_experiment.py \
    --holdout faceswap \
    --epochs 3 \
    --batch_size 16 \
    --num_workers 0 \
    --frequency_backbone resse \
    --hardened


In [ ]:
# LOTO Fold 4: NeuralTextures (~48 min)
%cd /kaggle/working/deepfake-detection
!git fetch origin && git reset --hard origin/main
!accelerate launch --multi_gpu --mixed_precision fp16 --num_processes 2 --main_process_port 29503 \
    scripts/train_loto_experiment.py \
    --holdout neuraltextures \
    --epochs 3 \
    --batch_size 16 \
    --num_workers 0 \
    --frequency_backbone resse \
    --hardened


In [ ]:
# LOTO Fold 5: Celeb-DF v2 (~18 min)
%cd /kaggle/working/deepfake-detection
!git fetch origin && git reset --hard origin/main
!accelerate launch --multi_gpu --mixed_precision fp16 --num_processes 2 --main_process_port 29504 \
    scripts/train_loto_experiment.py \
    --holdout celeb \
    --epochs 3 \
    --batch_size 16 \
    --num_workers 0 \
    --frequency_backbone resse \
    --hardened


In [ ]:
!python scripts/generate_benchmark_plots.py \
    --predictions /kaggle/working/test_predictions.json \
    --temporal_predictions /kaggle/working/temporal_test_predictions.json \
    --subdomain /kaggle/working/subdomain_results.json \
    --robustness /kaggle/working/robustness_results.json \
    --loto /kaggle/working/loto_results.json \
    --output_dir /kaggle/working/figures

!ls -lh /kaggle/working/*.pth /kaggle/working/*.json /kaggle/working/figures/*.png


In [ ]:
# Export trained backbone to ONNX
%cd /kaggle/working/deepfake-detection
!python scripts/export_onnx.py \
    --weights /kaggle/working/dual_stream_calibrated.pth \
    --output /kaggle/working/models/dual_stream.onnx \
    --img_size 256

# Benchmark Inference Latency & FPS
!python scripts/benchmark_latency.py \
    --weights /kaggle/working/dual_stream_calibrated.pth \
    --batch_size 32 \
    --device cuda

# Render 4-Panel Interpretability Diagnostics
!python scripts/visualize_attention_maps.py \
    --checkpoint /kaggle/working/dual_stream_calibrated.pth \
    --output_dir /kaggle/working/figures/attention_maps \
    --n_samples 6
